# A FAIRE POUR LANCER UN GROS ENTRAINEMENT

- utiliser un modèle b3 (ou +) que b0
- en csq changer le target size dans le preprocessor
- augmenter nombre d'epoch
- batch size ?

Optionnel :
- tester une autre loss

# **0. Librairies**

In [44]:
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader
from tqdm import tqdm
import sys
import os




sys.path.append(os.path.abspath(".."))

from lib.data.dataset import BeeDataset

from lib.data.preprocessing import TorchPreprocessor

from lib.data.train_val_split import train_val_split

from lib.data.preprocessing import TorchPreprocessor

from lib.data.data_augmentation import data_augmented_loader

In [45]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-3
num_classes = 50

notebook_dir = os.getcwd()

data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "data"))

## **1. Preprocessing**

In [46]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# preprocessor = TorchPreprocessor(
#     normalize=True,
#     mean = IMAGENET_MEAN,
#     std = IMAGENET_STD,
#     resize_method="pad",
#     target_size=(224, 224),
# )

# train_dataset, val_dataset = train_val_split(train_transform=preprocessor, val_transform=preprocessor)


In [47]:
import lib.data.preprocessing as prep
print(prep.__file__)

/home/alexandre-tonon/SDD/Hackathons/Hackaton_abeilles_tigres/lib/data/preprocessing.py


In [48]:
train_loader, val_loader = data_augmented_loader(IMAGENET_MEAN, IMAGENET_STD, target_size=(224, 224), batch_size=BATCH_SIZE)

Train prêt : 6417 images (avec augmentation ciblée)
Val prête  : 1582 images (sans augmentation)


## **2. Modèle**

In [49]:
def create_model(num_classes: int) -> nn.Module:
    model = models.efficientnet_b0(weights="IMAGENET1K_V1") #mettre b3 si ca marche
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, num_classes),
    )
    return model

model = create_model(num_classes)
model.to(DEVICE)

criterion = nn.CrossEntropyLoss()

# --- Variante ---
# pip install torchmetrics
# from torchmetrics.classification import MulticlassFocalLoss
# criterion = MulticlassFocalLoss(num_classes=num_classes, alpha=0.25, gamma=2.0)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)


## **3. F1-score**

In [50]:
import numpy as np

def compute_f1(all_labels, all_preds, num_classes):
    f1_per_class = []

    for cls in range(num_classes):
        # True Positives
        TP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) == cls))
        # False Positives
        FP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) != cls))
        # False Negatives
        FN = np.sum((np.array(all_preds) != cls) & (np.array(all_labels) == cls))

        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0

        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_per_class.append(f1)

    # F1 macro : moyenne des classes
    f1_macro = np.mean(f1_per_class)
    return f1_macro, f1_per_class

## **4. Fonctions de training et validation**

In [51]:
def train_one_epoch():
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    for images, labels in tqdm(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    # 🔹 Calcul F1 avec ta fonction existante
    f1_macro, f1_per_class = compute_f1(all_labels, all_preds, num_classes)

    # 🔹 Affichage
    # print(f"Train F1 macro: {f1_macro:.4f}")

    return total_loss / len(train_loader), correct / total, f1_macro, f1_per_class


In [52]:
import torch

def validate():
    model.eval()
    total_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calcul F1 manuel par classe
    f1_per_class = []
    for cls in range(num_classes):
        TP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) == cls))
        FP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) != cls))
        FN = np.sum((np.array(all_preds) != cls) & (np.array(all_labels) == cls))
        
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
        
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_per_class.append(f1)
    
    f1_macro = np.mean(f1_per_class)

    return total_loss / len(val_loader), f1_macro, f1_per_class

## **5. Entrainement**

**Vérif des labels**

In [53]:
# all_labels = [label for _, label in train_dataset]
# print("Label min:", min(all_labels))
# print("Label max:", max(all_labels))
# print("Nombre de classes uniques:", len(set(all_labels)))

# # Récupérer tous les labels uniques triés
# all_labels = sorted(set(label for _, label in train_dataset.samples))
# label_to_index = {label: i for i, label in enumerate(all_labels)}

# # Remapper les labels dans le dataset
# # for i, (path, label) in enumerate(train_dataset.samples):
# #     train_dataset.samples[i] = (path, label_to_index[label])

**Entrainement**

In [54]:
import csv
best_f1 = 0.0
best_model_path = "best_model.pth"

# Configuration du logging CSV
csv_file = "training_log.csv"
fieldnames = ['epoch', 'train_loss', 'train_acc', 'train_f1_macro', 'val_loss', 'val_f1_macro']

# Initialisation du fichier (écrase le précédent s'il existe)
with open(csv_file, mode='w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

for epoch in range(EPOCHS):
    train_loss, train_acc, train_f1_macro, train_f1_per_class = train_one_epoch()
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Train F1 Macro: {train_f1_macro:.4f}")
    print(f"Train F1 per class: {train_f1_per_class}")

    val_loss, val_f1_macro, val_f1_per_class = validate()
    print(f"Val   Loss: {val_loss:.4f}")
    print(f"Val   F1 Macro: {val_f1_macro:.4f}")
    print(f"Val   F1 per class: {val_f1_per_class}")

    with open(csv_file, mode='a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow({
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'train_f1_macro': train_f1_macro,
            'val_loss': val_loss,
            'val_f1_macro': val_f1_macro
        })

    # 🔹 Sauvegarde du meilleur modèle
    if val_f1_macro > best_f1:
        best_f1 = val_f1_macro
        torch.save(model.state_dict(), best_model_path)
        print(f" Nouveau meilleur modèle sauvegardé ! F1 Macro = {best_f1:.4f}")

100%|██████████| 201/201 [02:11<00:00,  1.53it/s]


Epoch 1/10
Train Loss: 1.5833 | Train Acc: 0.5900
Train F1 Macro: 0.5782
Train F1 per class: [np.float64(0.823076923076923), np.float64(0.7982456140350878), np.float64(0.23788546255506607), np.float64(0.6691729323308271), np.float64(0.6417910447761194), np.float64(0.4249084249084249), np.float64(0.6294820717131475), np.float64(0.7348242811501597), np.float64(0.8028673835125448), np.float64(0.5901639344262296), np.float64(0.8326848249027237), np.float64(0.6025104602510459), np.float64(0.35294117647058826), np.float64(0.7800829875518671), np.float64(0.40883977900552493), np.float64(0.3902439024390244), np.float64(0.648854961832061), np.float64(0.3389830508474576), np.float64(0.4109589041095891), np.float64(0.5876777251184834), np.float64(0.823943661971831), np.float64(0.5204460966542751), np.float64(0.5038759689922481), np.float64(0.7360594795539034), np.float64(0.23589743589743592), np.float64(0.4615384615384615), np.float64(0.8629032258064516), np.float64(0.7147335423197492), np.float

Val   Loss: 1.5988
Val   F1 Macro: 0.4362
Val   F1 per class: [0, np.float64(1.0), np.float64(0.4263322884012539), np.float64(0.5714285714285715), np.float64(0.3333333333333333), np.float64(0.6310160427807486), np.float64(0.4444444444444445), np.float64(1.0), np.float64(0.4), np.float64(0.6666666666666665), np.float64(1.0), np.float64(0.3448275862068966), np.float64(0.423841059602649), 0, np.float64(0.6511627906976744), np.float64(0.28571428571428575), 0, np.float64(0.5393258426966292), np.float64(0.5638766519823789), np.float64(0.851063829787234), np.float64(0.5), np.float64(0.23529411764705882), np.float64(0.6808510638297872), np.float64(0.75), np.float64(0.016260162601626018), np.float64(0.6846153846153845), np.float64(0.6666666666666666), 0, np.float64(0.288), 0, 0, np.float64(0.4), np.float64(0.06060606060606061), 0, np.float64(1.0), 0, np.float64(0.5544554455445545), np.float64(0.5), 0, np.float64(0.7692307692307693), np.float64(0.5853658536585366), np.float64(0.38202247191011235

100%|██████████| 201/201 [02:11<00:00,  1.52it/s]


Epoch 2/10
Train Loss: 0.7028 | Train Acc: 0.7910
Train F1 Macro: 0.7893
Train F1 per class: [np.float64(0.9491525423728814), np.float64(0.9747899159663865), np.float64(0.49295774647887325), np.float64(0.9007633587786259), np.float64(0.898876404494382), np.float64(0.6441947565543071), np.float64(0.844621513944223), np.float64(0.9314079422382672), np.float64(0.9662921348314607), np.float64(0.8326180257510729), np.float64(0.9747899159663866), np.float64(0.8416289592760181), np.float64(0.5037593984962406), np.float64(0.9365079365079365), np.float64(0.7067669172932332), np.float64(0.7929824561403509), np.float64(0.8537549407114625), np.float64(0.609053497942387), np.float64(0.5084745762711864), np.float64(0.7935222672064778), np.float64(0.9894736842105264), np.float64(0.7985865724381624), np.float64(0.7165354330708661), np.float64(0.8376068376068376), np.float64(0.451063829787234), np.float64(0.638655462184874), np.float64(0.9481481481481482), np.float64(0.9458483754512635), np.float64(0.

...


Epoch 9/10
Train Loss: 0.3089 | Train Acc: 0.9074
Train F1 Macro: 0.9078
Train F1 per class: [np.float64(0.9957081545064378), np.float64(1.0), np.float64(0.5882352941176471), np.float64(0.9831932773109243), np.float64(0.9779411764705882), np.float64(0.8), np.float64(0.9714285714285714), np.float64(0.9714285714285714), np.float64(0.9852941176470589), np.float64(0.9451476793248945), np.float64(0.9548872180451128), np.float64(0.9674796747967479), np.float64(0.7532467532467533), np.float64(0.972972972972973), np.float64(0.8791208791208792), np.float64(0.9111969111969112), np.float64(0.9658119658119659), np.float64(0.8634686346863469), np.float64(0.8375451263537906), np.float64(0.9136690647482014), np.float64(0.9874476987447698), np.float64(0.9122807017543859), np.float64(0.8394160583941606), np.float64(0.9492753623188407), np.float64(0.7330960854092526), np.float64(0.8671328671328671), np.float64(0.989010989010989), np.float64(1.0), np.float64(0.7265625), np.float64(0.9765258215962441), n

Val   Loss: 1.1707
Val   F1 Macro: 0.4588
Val   F1 per class: [0, 0, np.float64(0.4848484848484849), np.float64(0.4), np.float64(0.5), np.float64(0.5546218487394958), np.float64(0.36363636363636365), np.float64(0.5), np.float64(0.6666666666666666), np.float64(0.888888888888889), np.float64(0.6666666666666666), np.float64(0.5454545454545454), np.float64(0.5895953757225434), 0, np.float64(0.6842105263157895), np.float64(0.4615384615384615), np.float64(0.28571428571428575), np.float64(0.56), np.float64(0.729281767955801), np.float64(0.8888888888888888), 0, np.float64(0.3333333333333333), np.float64(0.6145251396648044), np.float64(0.8771929824561403), np.float64(0.6909090909090909), np.float64(0.8442906574394464), 0, 0, np.float64(0.5333333333333333), np.float64(0.6666666666666666), 0, np.float64(0.6666666666666666), np.float64(0.5409836065573771), 0, np.float64(0.6666666666666666), 0, np.float64(0.4153005464480874), np.float64(0.5555555555555556), 0, np.float64(0.5454545454545454), np.flo

100%|██████████| 201/201 [02:11<00:00,  1.52it/s]


Epoch 10/10
Train Loss: 0.2655 | Train Acc: 0.9196
Train F1 Macro: 0.9187
Train F1 per class: [np.float64(0.9856115107913669), np.float64(0.9962546816479401), np.float64(0.7404255319148936), np.float64(0.946969696969697), np.float64(0.9843749999999999), np.float64(0.8256227758007119), np.float64(0.9508196721311475), np.float64(0.9608540925266903), np.float64(0.9847328244274809), np.float64(0.9588014981273407), np.float64(1.0), np.float64(0.9545454545454546), np.float64(0.7326007326007326), np.float64(1.0), np.float64(0.9098712446351932), np.float64(0.9531914893617022), np.float64(0.9559748427672956), np.float64(0.8622222222222221), np.float64(0.8319327731092436), np.float64(0.9154929577464789), np.float64(0.9872340425531915), np.float64(0.9390681003584229), np.float64(0.8075471698113208), np.float64(0.9481481481481482), np.float64(0.7710843373493975), np.float64(0.8281250000000001), np.float64(0.9883268482490272), np.float64(0.9925925925925926), np.float64(0.8339768339768341), np.floa

Val   Loss: 0.9665
Val   F1 Macro: 0.5522
Val   F1 per class: [0, np.float64(1.0), np.float64(0.6041666666666666), 0, np.float64(0.8), np.float64(0.7325581395348836), np.float64(0.5714285714285715), 0, np.float64(0.6666666666666666), np.float64(0.5714285714285715), np.float64(1.0), np.float64(0.8), np.float64(0.6666666666666666), 0, np.float64(0.6829268292682926), np.float64(0.6666666666666666), 0, np.float64(0.5806451612903226), np.float64(0.7254901960784312), np.float64(0.9185185185185184), np.float64(1.0), np.float64(0.5454545454545454), np.float64(0.8349514563106796), np.float64(0.9056603773584904), np.float64(0.7659574468085107), np.float64(0.8452830188679246), np.float64(0.4), 0, np.float64(0.6029411764705882), np.float64(0.6666666666666666), 0, np.float64(0.6666666666666666), np.float64(0.5581395348837209), 0, np.float64(1.0), 0, np.float64(0.4615384615384615), np.float64(0.6), 0, np.float64(0.7142857142857143), np.float64(0.7854785478547855), np.float64(0.5263157894736843), 0, 

## **6. Création du fichier submission**

In [55]:
from torch.utils.data import DataLoader
import pandas as pd
import torch
from lib.data.dataset import BeeDataset

def submission(model, batch_size=32, transform=None, model_path="best_model.pth", save_path="submission.csv"):
    # Charger le modèle sur le bon device
    model.load_state_dict(torch.load(model_path, map_location=torch.device(DEVICE)))
    model.to(DEVICE)
    model.eval()
    
    # Dataset de test
    test_dataset = BeeDataset(train=False, transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    all_ids = []
    all_preds = []
    
    with torch.no_grad():
        for imgs, ids in test_loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            
            preds = torch.argmax(outputs, dim=1)
            
            # Convertir preds en int et ids en int ou str
            all_preds.extend(preds.cpu().tolist())
            all_ids.extend([int(x) if isinstance(x, torch.Tensor) else x for x in ids])
    
    submission_df = pd.DataFrame({
        "id": all_ids,
        "label": all_preds
    })
    
    submission_df.to_csv(save_path, index=False)
    print(f"Submission saved to {save_path}")

In [56]:
preprocessor = TorchPreprocessor(
    normalize=True,
    mean = IMAGENET_MEAN,
    std = IMAGENET_STD,
    resize_method="pad",
    target_size=(224, 224),
)


test_dataset = BeeDataset(train=False, transform=preprocessor)


submission(model, batch_size=32, transform=preprocessor, save_path="submission.csv")

/tmp/ipykernel_10022/2135808320.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=torch.device(DEVICE)))


Submission saved to submission.csv
